In [2]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


RAW_DATA_FILE = PROJECT_ROOT / "data" / "sample" / "crypto_market_sample.ndjson"

SCHEMA_FILE = PROJECT_ROOT / "schemas" / "crypto_market_schema.json"


print("Project Root :", PROJECT_ROOT)
print("Raw Data File:", RAW_DATA_FILE)
print("Schema File  :", SCHEMA_FILE)

Project Root : /home/diwakar1977/projects/crypto-market-intelligence-platform
Raw Data File: /home/diwakar1977/projects/crypto-market-intelligence-platform/data/sample/crypto_market_sample.ndjson
Schema File  : /home/diwakar1977/projects/crypto-market-intelligence-platform/schemas/crypto_market_schema.json


In [2]:
from src.spark.spark_session import SparkSessionFactory

spark = SparkSessionFactory.create()

print("Spark Version:", spark.version)

2026-08-26 20:17:17 - INFO - spark_session - ======================================================================
2026-08-26 20:17:17 - INFO - spark_session - SPARK SESSION INITIALIZATION
2026-08-26 20:17:17 - INFO - spark_session - ======================================================================
26/08/26 20:17:22 WARN Utils: Your hostname, LAPTOP-I2BK1TTC resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/26 20:17:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/26 20:17:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-08-26 20:17:27 - INFO - spark_session - SparkSession created successfully.application=crypto_market_intelligence master=local[*] version=3.5.6


Spark Version: 3.5.6


In [5]:
from src.schema.schema_loader import SchemaLoader

schema_loader = SchemaLoader()

schema = schema_loader.load(SCHEMA_FILE)

2026-08-26 20:23:36 - INFO - schema_loader - Starting schema loading from: /home/diwakar1977/projects/crypto-market-intelligence-platform/schemas/crypto_market_schema.json
2026-08-26 20:23:36 - INFO - schema_loader - Schema JSON loaded successfully from: /home/diwakar1977/projects/crypto-market-intelligence-platform/schemas/crypto_market_schema.json
2026-08-26 20:23:36 - INFO - schema_loader - Schema loaded successfully. Columns loaded: 26


In [9]:
df = (
    spark.read
    .schema(schema)
    .json(str(RAW_DATA_FILE))
)

print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 100
Columns: 26


In [13]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- name: string (nullable = true)
 |-- image: string (nullable = true)
 |-- current_price: double (nullable = true)
 |-- market_cap: long (nullable = true)
 |-- market_cap_rank: long (nullable = true)
 |-- fully_diluted_valuation: long (nullable = true)
 |-- total_volume: double (nullable = true)
 |-- high_24h: double (nullable = true)
 |-- low_24h: double (nullable = true)
 |-- price_change_24h: double (nullable = true)
 |-- price_change_percentage_24h: double (nullable = true)
 |-- market_cap_change_24h: double (nullable = true)
 |-- market_cap_change_percentage_24h: double (nullable = true)
 |-- circulating_supply: double (nullable = true)
 |-- total_supply: double (nullable = true)
 |-- max_supply: double (nullable = true)
 |-- ath: double (nullable = true)
 |-- ath_change_percentage: double (nullable = true)
 |-- ath_date: timestamp (nullable = true)
 |-- atl: double (nullable = true)
 |-- atl_change_pe

In [10]:
df.show(5, truncate=False)

+-----------+------+--------+-------------------------------------------------------------------------------------------+-------------+-------------+---------------+-----------------------+---------------+--------+--------+--------------------+---------------------------+----------------------+--------------------------------+---------------------+--------------------+----------+--------+---------------------+-------------------+----------+---------------------+-------------------+-------------------+----+
|id         |symbol|name    |image                                                                                      |current_price|market_cap   |market_cap_rank|fully_diluted_valuation|total_volume   |high_24h|low_24h |price_change_24h    |price_change_percentage_24h|market_cap_change_24h |market_cap_change_percentage_24h|circulating_supply   |total_supply        |max_supply|ath     |ath_change_percentage|ath_date           |atl       |atl_change_percentage|atl_date           |la

In [15]:
df.describe().show()

+-------+-----+------+--------+--------------------+-----------------+--------------------+------------------+-----------------------+--------------------+-----------------+-----------------+-------------------+---------------------------+---------------------+--------------------------------+--------------------+--------------------+--------------------+------------------+---------------------+------------------+---------------------+
|summary|   id|symbol|    name|               image|    current_price|          market_cap|   market_cap_rank|fully_diluted_valuation|        total_volume|         high_24h|          low_24h|   price_change_24h|price_change_percentage_24h|market_cap_change_24h|market_cap_change_percentage_24h|  circulating_supply|        total_supply|          max_supply|               ath|ath_change_percentage|               atl|atl_change_percentage|
+-------+-----+------+--------+--------------------+-----------------+--------------------+------------------+----------

In [16]:
drop_cols = ["roi"]

df = df.drop(*drop_cols)

In [17]:
duplicate_count = (
    df.groupby("id", "symbol")
    .count()
    .filter("count > 1")
    .show()
)

print("Duplicate Records:", duplicate_count)

df = df.dropDuplicates(["id", "symbol"])

+---+------+-----+
| id|symbol|count|
+---+------+-----+
+---+------+-----+

Duplicate Records: None


In [18]:
from pyspark.sql.functions import col, sum, when

null_counts = df.select(
    [
        sum(
            when(col(c).isNull(), 1)
            .otherwise(0)
        ).alias(c)
        for c in df.columns
    ]
)

null_counts.show()

+---+------+----+-----+-------------+----------+---------------+-----------------------+------------+--------+-------+----------------+---------------------------+---------------------+--------------------------------+------------------+------------+----------+---+---------------------+--------+---+---------------------+--------+------------+
| id|symbol|name|image|current_price|market_cap|market_cap_rank|fully_diluted_valuation|total_volume|high_24h|low_24h|price_change_24h|price_change_percentage_24h|market_cap_change_24h|market_cap_change_percentage_24h|circulating_supply|total_supply|max_supply|ath|ath_change_percentage|ath_date|atl|atl_change_percentage|atl_date|last_updated|
+---+------+----+-----+-------------+----------+---------------+-----------------------+------------+--------+-------+----------------+---------------------------+---------------------+--------------------------------+------------------+------------+----------+---+---------------------+--------+---+----------

In [19]:
from pyspark.sql.functions import col, count, when
from pyspark.sql.types import (
    DecimalType,
    DoubleType,
    FloatType,
    IntegerType,
    LongType,
    ShortType,
)

# Automaticlly detect numeric columns
numeric_columns = [
    field.name
    for field in df.schema.fields
    if isinstance(
        field.dataType,
        (
            IntegerType,
            LongType,
            DoubleType,
            FloatType,
            ShortType,
            DecimalType
        )
    )
]

print("Numeric Column:", numeric_columns)

# Print zero values in numeric columns
df.select(
    [
        count(
            when(col(column) == 0, column)
        ).alias(column)
        for column in numeric_columns
    ]
).show(truncate=False)

Numeric Column: ['current_price', 'market_cap', 'market_cap_rank', 'fully_diluted_valuation', 'total_volume', 'high_24h', 'low_24h', 'price_change_24h', 'price_change_percentage_24h', 'market_cap_change_24h', 'market_cap_change_percentage_24h', 'circulating_supply', 'total_supply', 'max_supply', 'ath', 'ath_change_percentage', 'atl', 'atl_change_percentage']


+-------------+----------+---------------+-----------------------+------------+--------+-------+----------------+---------------------------+---------------------+--------------------------------+------------------+------------+----------+---+---------------------+---+---------------------+
|current_price|market_cap|market_cap_rank|fully_diluted_valuation|total_volume|high_24h|low_24h|price_change_24h|price_change_percentage_24h|market_cap_change_24h|market_cap_change_percentage_24h|circulating_supply|total_supply|max_supply|ath|ath_change_percentage|atl|atl_change_percentage|
+-------------+----------+---------------+-----------------------+------------+--------+-------+----------------+---------------------------+---------------------+--------------------------------+------------------+------------+----------+---+---------------------+---+---------------------+
|0            |0         |0              |0                      |8           |0       |0      |1               |2          

In [21]:
from pyspark.sql.functions import col, round

round_columns = [
    "current_price",
    "high_24h",
    "low_24h",
    "price_change_24h",
    "price_change_percentage_24h",
    "market_cap_change_24h",
    "market_cap_change_percentage_24h",
    "ath",
    "ath_change_percentage",
    "atl",
    "atl_change_percentage"
]

for column in round_columns:
    df = df.withColumn(
        column,
        round(col(column) ,2)
    )

In [22]:
from pyspark.sql.functions import current_timestamp

df = df.withColumn(
    "ingest_timestamp",
    current_timestamp()
)

In [23]:
from pyspark.sql.functions import current_date, datediff

df = (
    df
    .withColumn(
        "days_since_ath",
        datediff(current_date(), "ath_date")
    )
    .withColumn(
        "days_since_atl",
        datediff(current_date(), "atl_date")
    )
)

In [24]:
df = df.withColumn(
    "daily_volatility_percentage",
    round(
        ((df.high_24h - df.low_24h) / df.current_price)
        , 2
    )
)

In [25]:
df = df.withColumn(
    "distance_from_ath",
    round(
        ((df.ath - df.current_price) / df.ath)
        ,2
    )
)

In [26]:
df = df.withColumn(
    "distance_from_atl",
    round(
        ((df.atl - df.current_price) / df.atl)
        ,2
    )
)

In [27]:
from pyspark.sql.functions import expr, round

df = df.withColumn(
    "volume_market_cap_ratio",
    round(
        expr("try_divide(total_volume, market_cap)")
        ,4
    )
)

In [28]:
df = df.withColumn(
    "supply_utilization_pct",
    round(
        df.circulating_supply / df.max_supply,
        2
    )
)

In [29]:
df = df.withColumn(
    "price_direction",
    when(df.price_change_24h > 0,"UP")
    .when(df.price_change_24h < 0,"DOWN")
    .otherwise("FLATE")
)

In [30]:
df.show(5,truncate=False)

+-----------+------+---------+---------------------------------------------------------------------------------------------------------+-------------+----------+---------------+-----------------------+------------+--------+-------+----------------+---------------------------+---------------------+--------------------------------+--------------------+-------------------+----------+------+---------------------+-------------------+-----+---------------------+-------------------+-------------------+--------------------------+--------------+--------------+---------------------------+-----------------+-----------------+-----------------------+----------------------+---------------+
|id         |symbol|name     |image                                                                                                    |current_price|market_cap|market_cap_rank|fully_diluted_valuation|total_volume|high_24h|low_24h|price_change_24h|price_change_percentage_24h|market_cap_change_24h|market_cap_change